# ProtocolHandler — Unified Facade

The single entry point for all MPMT protocol operations.  Created by
the server class (Composition Root) with all C++ instances injected.

Receives C++ injections → exposes ``prepare_join``, ``connect_leader``,
``aggregate``, ``query``, etc.  Application layer (Flask routes, tests)
talks exclusively through this object.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

from mpmt.protocol_handler import ProtocolHandler

handler = ProtocolHandler(
    party_id=0,
    rep3_inst=rep3_inst,
    tree_cache=tc,
    hash_seeds=hash_seeds,
    bf_size=bf_size,
    ch_prev=ch_prev,
    ch_next=ch_next,
    rep3_q_inst=rep3_q_inst,
    query_srv=query_srv,
)


## SetHolder Flow

Join / update protocol (two-phase HTTP):

1. ``handler.prepare_join(token)`` — insert placeholder into TreeCache
2. ``handler.three_way_confirm()`` — 1-byte ring handshake
3. ``handler.connect_leader(listen_port)`` — Leader receives BF shares
   from SetHolder via RingTransport
4. ``handler.connect_helper(port=port)`` — Helper joins temp Rep3 ring
   with SetHolder, receives BF share

Update reuses the same flow with ``handler.check_token(token, "update")``.


## Aggregation

- ``handler.aggregate()`` — executes the merge schedule using the
  persistent Rep3 ring.  Each merge step calls ``add_vec + hadamard + sub_vec``
  (paper Eq. 3.3).
- ``handler.do_quit(token)`` — heap-style removal from TreeCache
- ``handler.place_share(token, sv)`` — store a received share vector
  and mark ancestors dirty


## Query

``handler.query(element=..., ch_querier=..., hash_seeds=...)``
executes the full query protocol on this server.  Leader path:
hash → DPF → crng → reshare → dot → ET.  Helper path:
crng → reshare → dot → send dot share.


## Merge Formula (paper Eq. 3.3)

```
B(X ∪ Y) = add(X, Y) + hadamard(X, Y) - sub
```

Executed by ``handler.aggregate()`` for each step in the merge schedule.
